# AC OPF in JuMP (Self-Contained)
This notebook builds and solves a nonlinear AC Optimal Power Flow model directly in JuMP with Ipopt.

What this notebook does:
- Defines a 3-bus AC network in per-unit
- Implements exact rectangular-coordinate AC power-balance equations
- Solves OPF with generator and voltage limits
- Verifies KCL power-balance residuals numerically
- Summarizes which formulations are most convenient for multiphase distribution feeders

In [1]:
import Pkg

# Keep the notebook self-contained by activating the local project and ensuring
# the nonlinear optimization stack is present.
Pkg.activate(joinpath(@__DIR__, ".."))

required = ["JuMP", "Ipopt"]
project_deps = Set(String.(keys(Pkg.project().dependencies)))
for pkg in required
    if !(pkg in project_deps)
        Pkg.add(pkg)
    end
end

using JuMP
using Ipopt
using LinearAlgebra
using Printf

println("Environment ready")
println("JuMP + Ipopt loaded")

  Activating project at `c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl`


Environment ready
JuMP + Ipopt loaded


## Formulation used here (rectangular AC OPF)
We use an exact nonlinear AC OPF in rectangular coordinates.

For each bus $i$:
- Voltage state: $V_i = v_{r,i} + j v_{i,i}$
- Current injection: $I_i = \sum_j Y_{ij} V_j$
- Complex power injection: $S_i = P_i + jQ_i = V_i \overline{I_i}$

Using $Y_{ij} = G_{ij} + jB_{ij}$, define
$$
I_{r,i} = \sum_j \left(G_{ij} v_{r,j} - B_{ij} v_{i,j}\right), \quad
I_{i,i} = \sum_j \left(B_{ij} v_{r,j} + G_{ij} v_{i,j}\right)
$$
Then
$$
P_i = v_{r,i} I_{r,i} + v_{i,i} I_{i,i}, \quad
Q_i = v_{i,i} I_{r,i} - v_{r,i} I_{i,i}
$$
Power balance constraints are
$$
P_i = P_i^g - P_i^d, \quad Q_i = Q_i^g - Q_i^d
$$
with voltage-magnitude and generator-capability bounds.

Objective (quadratic generation cost):
$$
\min \sum_{g} \left(c_{2,g}(P_g)^2 + c_{1,g}P_g\right)
$$

Math cross-check references used while building this notebook:
- MATPOWER Standard AC OPF: https://matpower.app/manual/matpower/StandardACOPF.html
- PowerModels mathematical model (AC OPF and current-voltage variants): https://lanl-ansi.github.io/PowerModels.jl/stable/math-model/
- JuMP nonlinear modeling and constraints: https://jump.dev/JuMP.jl/stable/manual/nlp/

In [4]:
# 3-bus AC network in per-unit
branches = [
    (1, 2, 0.02 + 0.06im),
    (1, 3, 0.08 + 0.24im),
    (2, 3, 0.06 + 0.18im),
]

nb = 3
Y = zeros(ComplexF64, nb, nb)
for (f, t, z) in branches
    y = 1 / z
    Y[f, f] += y
    Y[t, t] += y
    Y[f, t] -= y
    Y[t, f] -= y
end
G = real.(Y)
B = imag.(Y)

# Active/reactive demand (pu)
Pd = Dict(1 => 0.0, 2 => 0.90, 3 => 1.00)
Qd = Dict(1 => 0.0, 2 => 0.30, 3 => 0.35)

function build_acopf_model(G, B, Pd, Qd)
    nb = size(G, 1)
    model = JuMP.Model(Ipopt.Optimizer)
    set_silent(model)

    @variable(model, vr[1:nb])
    @variable(model, vi[1:nb])

    # Generator dispatch variables
    @variable(model, 0.0 <= Pg1 <= 5.0)
    @variable(model, -5.0 <= Qg1 <= 5.0)
    @variable(model, 0.0 <= Pg2 <= 2.5)
    @variable(model, -2.0 <= Qg2 <= 2.0)

    # Slack/reference bus
    @constraint(model, vr[1] == 1.0)
    @constraint(model, vi[1] == 0.0)

    # Voltage limits on non-slack buses
    for i in 2:nb
        @constraint(model, 0.90^2 <= vr[i]^2 + vi[i]^2)
        @constraint(model, vr[i]^2 + vi[i]^2 <= 1.10^2)
    end

    # AC power-balance equations (rectangular coordinates)
    for i in 1:nb
        ir = @expression(model, sum(G[i, j] * vr[j] - B[i, j] * vi[j] for j in 1:nb))
        ii = @expression(model, sum(B[i, j] * vr[j] + G[i, j] * vi[j] for j in 1:nb))
        p_inj = @expression(model, vr[i] * ir + vi[i] * ii)
        q_inj = @expression(model, vi[i] * ir - vr[i] * ii)

        if i == 1
            @constraint(model, p_inj == Pg1 - Pd[i])
            @constraint(model, q_inj == Qg1 - Qd[i])
        elseif i == 2
            @constraint(model, p_inj == Pg2 - Pd[i])
            @constraint(model, q_inj == Qg2 - Qd[i])
        else
            @constraint(model, p_inj == -Pd[i])
            @constraint(model, q_inj == -Qd[i])
        end
    end

    # Quadratic generation cost
    @objective(model, Min, 5.0 * Pg1^2 + 20.0 * Pg1 + 8.0 * Pg2^2 + 10.0 * Pg2)

    return model, vr, vi, Pg1, Qg1, Pg2, Qg2
end

model, vr, vi, Pg1, Qg1, Pg2, Qg2 = build_acopf_model(G, B, Pd, Qd)
println("Model constructed")

Model constructed


In [5]:
optimize!(model)

term = termination_status(model)
prim = primal_status(model)
term_str = string(term)
prim_str = string(prim)

println("termination_status = ", term)
println("primal_status      = ", prim)

@assert term_str in ("OPTIMAL", "LOCALLY_SOLVED", "ALMOST_LOCALLY_SOLVED")
@assert prim_str in ("FEASIBLE_POINT", "NEARLY_FEASIBLE_POINT")

obj = objective_value(model)
@printf("objective          = %.8f\n", obj)
@printf("Pg1, Qg1           = %.6f, %.6f\n", value(Pg1), value(Qg1))
@printf("Pg2, Qg2           = %.6f, %.6f\n", value(Pg2), value(Qg2))

vm = [sqrt(value(vr[i])^2 + value(vi[i])^2) for i in 1:nb]
va_deg = [atan(value(vi[i]), value(vr[i])) * 180 / pi for i in 1:nb]

for i in 1:nb
    @printf("bus %d: Vm=%.6f pu, Va=%.4f deg\n", i, vm[i], va_deg[i])
end

termination_status = LOCALLY_SOLVED
primal_status      = FEASIBLE_POINT
objective          = 41.22407154
Pg1, Qg1           = 0.799479, 0.125492
Pg2, Qg2           = 1.148543, 0.668576
bus 1: Vm=1.000000 pu, Va=0.0000 deg
bus 2: Vm=0.998668 pu, Va=-1.1534 deg
bus 3: Vm=0.917682 pu, Va=-6.3454 deg


In [6]:
# Numerical verification of nodal power-balance equations.
max_p_mismatch = 0.0
max_q_mismatch = 0.0

for i in 1:nb
    vr_i = value(vr[i])
    vi_i = value(vi[i])
    ir_i = sum(G[i, j] * value(vr[j]) - B[i, j] * value(vi[j]) for j in 1:nb)
    ii_i = sum(B[i, j] * value(vr[j]) + G[i, j] * value(vi[j]) for j in 1:nb)

    p_lhs = vr_i * ir_i + vi_i * ii_i
    q_lhs = vi_i * ir_i - vr_i * ii_i

    p_rhs = i == 1 ? value(Pg1) - Pd[i] : (i == 2 ? value(Pg2) - Pd[i] : -Pd[i])
    q_rhs = i == 1 ? value(Qg1) - Qd[i] : (i == 2 ? value(Qg2) - Qd[i] : -Qd[i])

    max_p_mismatch = max(max_p_mismatch, abs(p_lhs - p_rhs))
    max_q_mismatch = max(max_q_mismatch, abs(q_lhs - q_rhs))
end

@printf("max |P mismatch|   = %.3e\n", max_p_mismatch)
@printf("max |Q mismatch|   = %.3e\n", max_q_mismatch)

@assert max_p_mismatch <= 1e-6
@assert max_q_mismatch <= 1e-6

total_pg = value(Pg1) + value(Pg2)
total_pd = sum(values(Pd))
total_qg = value(Qg1) + value(Qg2)
total_qd = sum(values(Qd))

@printf("total Pg           = %.6f pu\n", total_pg)
@printf("total Pd           = %.6f pu\n", total_pd)
@printf("total active loss  = %.6f pu\n", total_pg - total_pd)
@printf("total Qg           = %.6f pu\n", total_qg)
@printf("total Qd           = %.6f pu\n", total_qd)

max |P mismatch|   = 1.315e-13
max |Q mismatch|   = 4.260e-13
total Pg           = 1.948023 pu
total Pd           = 1.900000 pu
total active loss  = 0.048023 pu
total Qg           = 0.794068 pu
total Qd           = 0.650000 pu


## Recommended formulations for your multiphase feeder case
For unbalanced distribution OPF (IEEE13/37/123 with regulators/transformers), these are practical choices:

1. ACRU (rectangular voltage)
- Good default for multiphase OPF in PMD
- Usually easier to debug than polar angle formulations
- Start with Kron-reduced, phase-projected data

2. IVRU (current-voltage)
- Useful when current-balance representation is numerically preferable
- Can be more robust for some difficult network physics, but data consistency is important

3. ACPU (polar voltage)
- Mathematically equivalent AC model, but often trickier numerically on unbalanced feeders

Suggested PMD parse/solve starting point for your research pipeline:

```julia
data = PowerModelsDistribution.parse_file(
    dss_path;
    data_model = PowerModelsDistribution.MATHEMATICAL,
    make_pu = true,
    kron_reduce = true,
    phase_project = true,
)
result = PowerModelsDistribution.solve_mc_opf(
    data,
    PowerModelsDistribution.ACRUPowerModel,
    Ipopt.Optimizer,
 )
```

Validation checklist before trusting results:
- Check termination_status and primal_status
- Verify max nodal P/Q mismatch
- Verify all voltage bounds and generator bounds
- Compare voltage profiles against OpenDSS or a trusted baseline

## 13-bus AC OPF with additional engineering constraints
This appended section provides a self-contained 13-bus AC OPF in JuMP and explicitly includes:
- Branch current magnitude constraints
- Branch angle-difference constraints
- Substation active/reactive/apparent power constraints

Modeling notes:
- Polar AC power-flow equations are used for direct angle-difference bounds.
- This is a compact 13-bus radial test system for formulation validation.
- All constraints are checked numerically after solving.

In [2]:
# 13-bus radial test network (per-unit, single-phase compact AC OPF benchmark).
nb13 = 13
branches13 = [
    (1, 2, 0.010 + 0.030im, 2.0),
    (2, 3, 0.012 + 0.036im, 1.8),
    (3, 4, 0.010 + 0.030im, 1.6),
    (4, 5, 0.015 + 0.045im, 1.5),
    (5, 6, 0.012 + 0.040im, 1.3),
    (3, 7, 0.020 + 0.060im, 1.2),
    (7, 8, 0.015 + 0.050im, 1.0),
    (8, 9, 0.015 + 0.050im, 1.0),
    (2, 10, 0.020 + 0.060im, 1.2),
    (10, 11, 0.015 + 0.045im, 1.1),
    (11, 12, 0.015 + 0.045im, 1.0),
    (12, 13, 0.020 + 0.060im, 0.9),
]

Y13 = zeros(ComplexF64, nb13, nb13)
for (f, t, z, _) in branches13
    y = 1 / z
    Y13[f, f] += y
    Y13[t, t] += y
    Y13[f, t] -= y
    Y13[t, f] -= y
end
G13 = real.(Y13)
B13 = imag.(Y13)

Pd13 = zeros(nb13)
Qd13 = zeros(nb13)
for (i, p, q) in [
    (2, 0.18, 0.07), (3, 0.15, 0.06), (4, 0.12, 0.05), (5, 0.10, 0.04),
    (6, 0.08, 0.03), (7, 0.13, 0.05), (8, 0.10, 0.04), (9, 0.09, 0.04),
    (10, 0.11, 0.04), (11, 0.09, 0.04), (12, 0.08, 0.03), (13, 0.07, 0.03),
]
    Pd13[i] = p
    Qd13[i] = q
end

function build_13bus_constrained_acopf(; delta_deg = 35.0, s_sub_max = 2.2)
    model13 = JuMP.Model(Ipopt.Optimizer)
    set_silent(model13)

    @variable(model13, Vm13[1:nb13])
    @variable(model13, Va13[1:nb13])

    # Substation power constraints at bus 1 (P/Q bounds + apparent power cap).
    @variable(model13, 0.0 <= Pg_sub13 <= 2.0)
    @variable(model13, -1.0 <= Qg_sub13 <= 1.0)
    @constraint(model13, Pg_sub13^2 + Qg_sub13^2 <= s_sub_max^2)

    # Two downstream DERs.
    @variable(model13, 0.0 <= Pg7_13 <= 0.25)
    @variable(model13, -0.15 <= Qg7_13 <= 0.15)
    @variable(model13, 0.0 <= Pg12_13 <= 0.20)
    @variable(model13, -0.12 <= Qg12_13 <= 0.12)

    # Slack/reference bus and voltage limits.
    @constraint(model13, Vm13[1] == 1.0)
    @constraint(model13, Va13[1] == 0.0)
    for i in 2:nb13
        @constraint(model13, 0.93 <= Vm13[i] <= 1.07)
    end

    # Angle-difference constraints on each branch.
    delta_max13 = deg2rad(delta_deg)
    for (f, t, _, _) in branches13
        @constraint(model13, -delta_max13 <= Va13[f] - Va13[t] <= delta_max13)
    end

    # AC nodal power balance in polar coordinates.
    for i in 1:nb13
        p_inj = @expression(model13, sum(Vm13[i] * Vm13[j] * (G13[i, j] * cos(Va13[i] - Va13[j]) + B13[i, j] * sin(Va13[i] - Va13[j])) for j in 1:nb13))
        q_inj = @expression(model13, sum(Vm13[i] * Vm13[j] * (G13[i, j] * sin(Va13[i] - Va13[j]) - B13[i, j] * cos(Va13[i] - Va13[j])) for j in 1:nb13))

        p_gen = i == 1 ? Pg_sub13 : (i == 7 ? Pg7_13 : (i == 12 ? Pg12_13 : 0.0))
        q_gen = i == 1 ? Qg_sub13 : (i == 7 ? Qg7_13 : (i == 12 ? Qg12_13 : 0.0))

        @constraint(model13, p_inj == p_gen - Pd13[i])
        @constraint(model13, q_inj == q_gen - Qd13[i])
    end

    # Branch current constraints: |I_ft|^2 <= Imax^2.
    for (f, t, z, imax) in branches13
        y = 1 / z
        yabs2 = abs2(y)
        @constraint(model13, yabs2 * (Vm13[f]^2 + Vm13[t]^2 - 2 * Vm13[f] * Vm13[t] * cos(Va13[f] - Va13[t])) <= imax^2)
    end

    @objective(model13, Min, 25.0 * Pg_sub13^2 + 5.0 * Pg_sub13 + 6.0 * Pg7_13^2 + 6.0 * Pg12_13^2)

    # Warm-start improves robustness for this nonlinear AC model.
    set_start_value.(Vm13, 1.0)
    set_start_value.(Va13, 0.0)
    set_start_value(Pg_sub13, sum(Pd13))
    set_start_value(Qg_sub13, sum(Qd13))
    set_start_value(Pg7_13, 0.10)
    set_start_value(Qg7_13, 0.0)
    set_start_value(Pg12_13, 0.08)
    set_start_value(Qg12_13, 0.0)

    return model13, Vm13, Va13, Pg_sub13, Qg_sub13, Pg7_13, Qg7_13, Pg12_13, Qg12_13, delta_max13, s_sub_max
end

model13, Vm13, Va13, Pg_sub13, Qg_sub13, Pg7_13, Qg7_13, Pg12_13, Qg12_13, delta_max13, s_sub_max13 = build_13bus_constrained_acopf()
optimize!(model13)

term13 = termination_status(model13)
prim13 = primal_status(model13)
println("13-bus termination_status = ", term13)
println("13-bus primal_status      = ", prim13)
@assert string(term13) in ("OPTIMAL", "LOCALLY_SOLVED", "ALMOST_LOCALLY_SOLVED")
@assert string(prim13) in ("FEASIBLE_POINT", "NEARLY_FEASIBLE_POINT")

@printf("13-bus objective           = %.8f\n", objective_value(model13))
@printf("substation Pg,Qg          = %.6f, %.6f\n", value(Pg_sub13), value(Qg_sub13))
@printf("DER@7 Pg,Qg               = %.6f, %.6f\n", value(Pg7_13), value(Qg7_13))
@printf("DER@12 Pg,Qg              = %.6f, %.6f\n", value(Pg12_13), value(Qg12_13))

vm13 = [value(Vm13[i]) for i in 1:nb13]
@printf("min/max Vm                = %.6f / %.6f pu\n", minimum(vm13), maximum(vm13))


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

13-bus termination_status = LOCALLY_SOLVED
13-bus primal_status      = FEASIBLE_POINT
13-bus objective           = 23.68262115
substation Pg,Qg          = 0.865766, 0.297615
DER@7 Pg,Qg               = 0.250000, 0.150000
DER@12 Pg,Qg              = 0.200000, 0.120000
min/max Vm                = 0.954407 / 1.000000 pu


In [3]:
# Numerical validation: power-balance residuals + explicit constraint checks.
let
    max_p_mismatch13 = 0.0
    max_q_mismatch13 = 0.0

    for i in 1:nb13
        p_lhs = sum(
            value(Vm13[i]) * value(Vm13[j]) * (
                G13[i, j] * cos(value(Va13[i]) - value(Va13[j])) +
                B13[i, j] * sin(value(Va13[i]) - value(Va13[j]))
            ) for j in 1:nb13
        )
        q_lhs = sum(
            value(Vm13[i]) * value(Vm13[j]) * (
                G13[i, j] * sin(value(Va13[i]) - value(Va13[j])) -
                B13[i, j] * cos(value(Va13[i]) - value(Va13[j]))
            ) for j in 1:nb13
        )

        p_gen = i == 1 ? value(Pg_sub13) : (i == 7 ? value(Pg7_13) : (i == 12 ? value(Pg12_13) : 0.0))
        q_gen = i == 1 ? value(Qg_sub13) : (i == 7 ? value(Qg7_13) : (i == 12 ? value(Qg12_13) : 0.0))

        p_rhs = p_gen - Pd13[i]
        q_rhs = q_gen - Qd13[i]

        max_p_mismatch13 = max(max_p_mismatch13, abs(p_lhs - p_rhs))
        max_q_mismatch13 = max(max_q_mismatch13, abs(q_lhs - q_rhs))
    end

    max_angle_diff13 = 0.0
    max_current_ratio13 = 0.0
    for (f, t, z, imax) in branches13
        delta = abs(value(Va13[f]) - value(Va13[t]))
        max_angle_diff13 = max(max_angle_diff13, delta)

        y = 1 / z
        If_t = abs(y * ((value(Vm13[f]) * cis(value(Va13[f]))) - (value(Vm13[t]) * cis(value(Va13[t])))))
        max_current_ratio13 = max(max_current_ratio13, If_t / imax)
    end

    substation_s_ratio13 = sqrt(value(Pg_sub13)^2 + value(Qg_sub13)^2) / s_sub_max13

    @printf("13-bus max |P mismatch|    = %.3e\n", max_p_mismatch13)
    @printf("13-bus max |Q mismatch|    = %.3e\n", max_q_mismatch13)
    @printf("13-bus max |angle diff|    = %.4f deg\n", max_angle_diff13 * 180 / pi)
    @printf("13-bus max current usage   = %.4f pu of branch limits\n", max_current_ratio13)
    @printf("13-bus substation S usage  = %.4f pu of S limit\n", substation_s_ratio13)

    @assert max_p_mismatch13 <= 1e-6
    @assert max_q_mismatch13 <= 1e-6
    @assert max_angle_diff13 <= delta_max13 + 1e-8
    @assert max_current_ratio13 <= 1.0 + 1e-8
    @assert substation_s_ratio13 <= 1.0 + 1e-8
end

13-bus max |P mismatch|    = 2.096e-15
13-bus max |Q mismatch|    = 8.382e-15
13-bus max |angle diff|    = 1.3410 deg
13-bus max current usage   = 0.4577 pu of branch limits
13-bus substation S usage  = 0.4161 pu of S limit
